# ML-10 - Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROHITCRAFTSYT/flyrank-int/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

A model score is not the product. This notebook turns my **validated output** into a human-reviewed
**content action playbook** with known limits - the version a content editor could actually work
from, and the exact files next week's paper builds on.

**What "validated output" means here, honestly (the claim ladder, ML-09):** across ML-07/08/09 the
thing that survived was the **rule**, not a learned model. The ML-07 position-adjusted CTR-deficit
score ranked "which pages to review first"; ML-08 showed a model did **not** beat it at the one-week
operating point (precision@50 tie, ~0.84 vs a 0.53 base rate, client-grouped); ML-09 confirmed the
split was honest and the features leak-free. So the playbook is built on that rule. The **queue
ORDER** is validated decision-support. The **archetype ACTION** (rewrite / refresh / expand /
promote) is *directional guidance* from the page's own signals plus the paper's observed
portfolio patterns - not a validated per-page prediction. I keep those two claim levels separate
everywhere below.

Runs top-to-bottom on the in-repo starter slice (no token). Exports at the end.

## 1. Ranked actions + reason codes

### The queue, in one sentence
> Review visible pages in order of **how many clicks they leave on the table versus same-rank
> peers** (`missed_clicks_90d`); for each, the **archetype** says *what kind of fix* and the
> **reason code** says *why it's flagged*.

### Archetype -> action -> reason-code mapping
Every actionable page is assigned to exactly one archetype by a documented cascade (checked in
this order; first match wins). The action is the recommended *fix type*; the reason code is the
one-line justification a human can audit.

| # | Archetype | Trigger (on the visible, clicks>=1 universe) | Reason code | Recommended action | Auto? |
|---|---|---|---|---|---|
| 0 | `verify_tracking` | CTR < 0.03pp on >=50k impressions (implausible) | `possible_tracking_gap` | **Verify click tracking BEFORE any content work** | **No - human only** |
| 1 | `stale_refresh` | under-capturing **and** age>=180d **and** >=90d since update | `mature_past_fresh_window` | Refresh: update facts/dates, expand thin sections | Draft yes, publish no |
| 2 | `thin_expand` | under-capturing **and** word_count < 1500 | `thin_but_visible` | Expand depth where demand already exists | Draft yes, publish no |
| 3 | `ctr_rewrite` | under-capturing, adequately fresh + deep | `under_capturing_vs_rank` | Rewrite title & meta to lift CTR | Draft yes, publish no |
| 4 | `striking_distance` | not under-capturing, avg_position 11-20 | `striking_distance` | Improve relevance + internal links toward page 1 | Draft yes, publish no |
| - | `monitor` | at/above expected CTR, page 1, fresh | `at_or_above_expected` | No action this cycle; watch | n/a |

**The decay/refresh insight, stated honestly.** The paper's strongest portfolio-wide lever is
refreshing mature pages before they decay (Findings #2/#4/#8). In *my* CTR-opportunity universe
that lever is real but **secondary**, and here's why, measured rather than assumed: my visible
pages are **old but freshly maintained** - median age ~236 days yet median ~22 days since last
update. Most under-capturing pages have already been refreshed recently, so a rewrite (not a
refresh) is usually the missing move. `stale_refresh` still captures the genuine mature-and-drifting
minority; I'm just not going to overclaim refresh as the headline for this lane when the data says
these pages are mostly fresh. (This is the paper's "Old + Refreshed" quadrant showing up in my slice.)

### Cost / value lens (secondary to the rank, on purpose)
Following the paper's defensible formula (**clicks x CPC, never impressions x CPC**), I attach
`est_value_usd = missed_clicks_90d x cpc` - the click-equivalent value at stake if a page recovered
to its peers' CTR. Two honest limits keep this a *secondary* column, not the primary sort:
CPC is known for only ~23% of the queue, and CPC is a benchmark, **not booked revenue**. So the
queue ranks by `missed_clicks_90d` (validated, universal) and shows value **where CPC exists**.
Cost side: budget ~1 editor-hour per page; the value column is what lets a lead spend the scarcest
hours on the highest-value pages first.

In [1]:
# --- Build the playbook queue on the validated ML-07 universe ----------------------
import os, sys, subprocess
import numpy as np, pandas as pd
pd.set_option("display.width", 200)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL, REPO_DIR = "https://github.com/flyrank-bih/flyrank-ml-internship-starter", "flyrank-ml-internship-starter"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv").drop_duplicates("content_id")

# Visible universe + clicks>=1 floor (ML-07). One row = one page an editor could open this week.
vis = df[(df.impressions_90d >= 500) & (df.avg_position > 0) & (df.avg_position <= 20) &
         (df.content_age_days >= 90) & (df.clicks_90d >= 1)].copy()
vis["pos_band"]     = vis.avg_position.round().clip(1, 20).astype(int)
vis["expected_ctr"] = vis.groupby("pos_band")["ctr"].transform("median")   # the CTR-vs-rank curve
vis["ctr_gap"]      = vis.ctr - vis.expected_ctr
vis["missed_clicks_90d"] = np.where(vis.ctr_gap < 0, (vis.expected_ctr - vis.ctr) / 100 * vis.impressions_90d, 0.0)
vis["est_value_usd"]     = (vis.missed_clicks_90d * vis.cpc.fillna(0)).round(2)   # clicks x CPC (paper's formula)
band_n = vis.groupby("pos_band")["ctr"].transform("size")
vis["thin_band_flag"]    = (band_n < 100).astype(int)                              # Signal-1 unreliable bands (ML-07)

# --- Archetype cascade: first match wins (documented order) ------------------------
ACTIONS = {
    "verify_tracking":   ("possible_tracking_gap",    "Verify click tracking BEFORE any content work"),
    "stale_refresh":     ("mature_past_fresh_window", "Refresh: update facts/dates, expand thin sections"),
    "thin_expand":       ("thin_but_visible",         "Expand depth where demand already exists"),
    "ctr_rewrite":       ("under_capturing_vs_rank",  "Rewrite title & meta to lift CTR"),
    "striking_distance": ("striking_distance",        "Improve relevance + internal links toward page 1"),
    "monitor":           ("at_or_above_expected",     "No action this cycle; watch"),
}
def archetype(r):
    if r.ctr < 0.03 and r.impressions_90d >= 50_000:          return "verify_tracking"
    if r.missed_clicks_90d <= 0:
        return "striking_distance" if r.avg_position >= 11 else "monitor"
    if r.content_age_days >= 180 and r.days_since_last_update >= 90: return "stale_refresh"
    if pd.notna(r.word_count) and r.word_count < 1500:          return "thin_expand"
    return "ctr_rewrite"

vis["archetype"]   = vis.apply(archetype, axis=1)
vis["reason_code"] = vis.archetype.map(lambda a: ACTIONS[a][0])
vis["action"]      = vis.archetype.map(lambda a: ACTIONS[a][1])

# Review policy columns (the DISCUSSION is section 3; computed here so the queue carries them).
REVIEW_NOTE = {
    "verify_tracking":   "STOP: confirm click tracking in analytics before any content work",
    "stale_refresh":     "confirm facts are actually dated before rewriting",
    "thin_expand":       "confirm the topic needs depth; do not pad to a word target",
    "ctr_rewrite":       "confirm title/snippet is weak for the query intent (not a hard query)",
    "striking_distance": "confirm real demand + topical fit before spending link equity",
}
vis["human_review_required"] = True                            # nothing here auto-publishes (section 3)
vis["auto_publish"]          = False                           # hard rule for every row
vis["review_note"]           = vis.archetype.map(REVIEW_NOTE).fillna("")
vis["no_go_verify_first"]    = (vis.archetype == "verify_tracking")

print(f"universe: {len(vis):,} visible pages across {vis.client_id.nunique()} clients\n")
print("=== queue composition by archetype ===")
comp = (vis.groupby("archetype")
           .agg(pages=("content_id", "size"),
                total_missed_clicks=("missed_clicks_90d", "sum"),
                value_usd_where_cpc_known=("est_value_usd", "sum"))
           .reindex(ACTIONS.keys()).fillna(0).round(0).astype({"pages": int}))
print(comp.to_string())

# --- The ranked action queue: primary sort missed_clicks, then impressions ---------
queue = (vis[vis.archetype != "monitor"]
         .sort_values(["missed_clicks_90d", "impressions_90d"], ascending=False)
         .reset_index(drop=True))
queue.insert(0, "rank", queue.index + 1)
print(f"\nactionable queue: {len(queue):,} pages ({len(vis)-len(queue):,} held in 'monitor')")
print("\n=== top 15 of the ranked queue ===")
show = ["rank", "content_id", "archetype", "reason_code", "missed_clicks_90d", "est_value_usd",
        "cpc", "avg_position", "impressions_90d", "ctr", "expected_ctr", "thin_band_flag"]
print(queue[show].head(15).round({"missed_clicks_90d": 0, "ctr": 2, "expected_ctr": 2, "cpc": 2}).to_string(index=False))
print("\nNote the top of the queue already contains a few `verify_tracking` rows - highest apparent")
print("opportunity, but they must be tracking-checked first, not rewritten (see the no-go list, section 3).")


universe: 10,808 visible pages across 28 clients

=== queue composition by archetype ===
                   pages  total_missed_clicks  value_usd_where_cpc_known
archetype                                                               
verify_tracking        8               1707.0                      528.0
stale_refresh       1415              18403.0                     6686.0
thin_expand          106                584.0                      419.0
ctr_rewrite         3778              46059.0                    23731.0
striking_distance   1655                  0.0                        0.0
monitor             3846                  0.0                        0.0

actionable queue: 6,962 pages (3,846 held in 'monitor')

=== top 15 of the ranked queue ===
 rank           content_id       archetype              reason_code  missed_clicks_90d  est_value_usd   cpc  avg_position  impressions_90d  ctr  expected_ctr  thin_band_flag
    1 content_5fe46e04994d   stale_refresh mature_past_fresh

## 2. Intended use and limits

**Who uses this, for what.** A content editor or SEO lead with ~50 review-hours a week, deciding
*which already-visible pages to open first* and *what kind of fix each likely needs*. It is a
**prioritisation aid**, decision-support - it points attention; the human decides and executes.

**Where it is valid.** One row = one visible page (>=500 impressions/90d, average position 1-20,
>=90 days old, >=1 click). Inside that universe, on this portfolio, in this period. The queue
*order* is the part with validation behind it (ML-08: precision@50 ~0.84 vs 0.53 base rate,
client-grouped; a learned model did not beat it).

**Where it stops being valid (limits, each measured not asserted):**
- **Not causal.** "Worth reviewing first" is decision-support, not "rewriting will raise CTR."
  Cross-sectional snapshot, no intervention - the claim ladder forbids the causal form.
- **Position confound (the big one).** CTR next month can rise because a page *ranked* better, not
  because anyone improved it; the starter slice has no per-window rank to hold constant (ML-08/09).
- **Archetype action is directional.** The fix *type* comes from page signals + the paper's
  observed patterns, not a per-page validated model. Two editors may reasonably reassign an archetype.
- **CPC coverage ~23%.** The value column is blank for most pages and is a benchmark, not revenue.
- **Thin-band expected_ctr.** Position bands 1-2 rest on few pages (`thin_band_flag`); ranking there
  is shaky (ML-07 Signal 1 = MIXED).
- **28 clients.** The grouped-split gap was small, but this is not a large or random panel of sites.
- **Not an SEO-algorithm claim.** This models outcomes in one portfolio; it does not describe Google.

In [2]:
# --- Quantify the limits so the caveats carry numbers, not adjectives --------------
print("=== the limits, measured ===")
print(f"  universe coverage      : {len(vis):,} of {len(df):,} pages ({len(vis)/len(df):.0%}) meet the visible+floor rule")
print(f"  clients represented    : {vis.client_id.nunique()} (small panel)")
print(f"  CPC known (value usable): {(vis.cpc>0).mean():.0%} of the queue; value is blank for the rest")
print(f"  thin-band rows in queue : {int(queue.thin_band_flag.sum())} of {len(queue):,} sit in a <100-page band (shaky expected_ctr)")
print(f"  validated at            : precision@50 ~0.84 vs base rate ~0.53 (ML-08, client-grouped, forward proxy)")
print(f"  NOT validated           : the per-page archetype action, and any causal 'rewriting -> +CTR' reading")
print(f"  age vs freshness        : median age {vis.content_age_days.median():.0f}d, median days-since-update "
      f"{vis.days_since_last_update.median():.0f}d -> old-but-fresh (why refresh is a secondary lever here)")


=== the limits, measured ===
  universe coverage      : 10,808 of 30,000 pages (36%) meet the visible+floor rule
  clients represented    : 28 (small panel)
  CPC known (value usable): 22% of the queue; value is blank for the rest
  thin-band rows in queue : 12 of 6,962 sit in a <100-page band (shaky expected_ctr)
  validated at            : precision@50 ~0.84 vs base rate ~0.53 (ML-08, client-grouped, forward proxy)
  NOT validated           : the per-page archetype action, and any causal 'rewriting -> +CTR' reading
  age vs freshness        : median age 236d, median days-since-update 22d -> old-but-fresh (why refresh is a secondary lever here)


## 3. Human review + the no-go list

### What a person must check before acting (per archetype)
- **Every row:** does the page's actual title/snippet look weak for its query intent? The score
  says *under-capturing*; only a human confirms the *cause* is fixable copy vs a hard query.
- **`verify_tracking`:** open analytics first. If clicks are simply mis-tracked, there is no content
  problem - do not touch the page.
- **`stale_refresh`:** confirm the facts are actually dated before rewriting history that's still correct.
- **`thin_expand`:** confirm the topic genuinely needs more depth - do not pad to a word target.
- **`striking_distance`:** confirm real search demand and topical fit before spending link equity.

### The no-go list - what should NOT be automated
1. **Auto-publishing any change.** Drafts only; a human approves every edit that ships.
2. **Acting on `verify_tracking` pages as content.** They may be measurement artifacts (ML-03/09),
   not weak content. Fix tracking, then re-queue.
3. **Treating the archetype action as a prediction.** It is directional; only the *ranking* is validated.
4. **Ranking by `est_value_usd` alone.** CPC is sparse and a benchmark; it would silently drop the ~77% of pages with no CPC.
5. **Trusting `thin_band_flag` rows' order.** Bands 1-2 are unreliable; use as a loose hint.
6. **Applying outside the universe** (no position data, <500 impressions, zero clicks) - out of scope by design.
7. **Bulk edits across a whole client** off one score - the panel is 28 clients; generalisation is untested at scale.
8. **Framing any of this as "what Google rewards."** It models one portfolio's outcomes, full stop.

In [3]:
# --- Summarize the human-review + no-go structure (columns built in section 1) -----
print("=== human-review gates ===")
print(f"  rows requiring human review before acting : {int(vis.human_review_required.sum())} of {len(vis):,} (all)")
print(f"  rows auto-publishable                     : {int(vis.auto_publish.sum())} (zero, by rule)")
print(f"  NO-GO verify-first (tracking suspect)     : {int(vis.no_go_verify_first.sum())}")
print(f"  of which sit in the top 50 by opportunity : {int(queue.head(50).archetype.eq('verify_tracking').sum())}"
      "  <- exactly why publish is never automated")


=== human-review gates ===
  rows requiring human review before acting : 10808 of 10,808 (all)
  rows auto-publishable                     : 0 (zero, by rule)
  NO-GO verify-first (tracking suspect)     : 8


  of which sit in the top 50 by opportunity : 5  <- exactly why publish is never automated


## 4. Monitoring / retrain triggers

The winner is a **rule**, so "retrain" is light: re-fit the position-band CTR curve and re-run the
ML-08 comparison; only escalate to a warehouse-trained model if the rule's precision decays. I store
a **reference snapshot** now (section 5's JSON) so future runs can diff against it.

**Monitor monthly - trigger review if:**
| Signal | Why it matters | Trigger |
|---|---|---|
| CTR-vs-position curve (`expected_ctr` by band) | the entire rule is built on it | any band's median CTR moves > ~25% |
| Actionable fraction / base rate | queue size drift = universe shift | actionable share moves > ~10pts from ~64% |
| `verify_tracking` count | rising = tracking regressions upstream | count grows materially month-over-month |
| precision@50 on a **fresh forward month** | the only real check of usefulness | drops toward the base rate (~0.53) |
| Client coverage | silent loss of a client's data | fewer clients present than last run |

**Retrain / re-fit triggers:** re-fit the band curve every month (cheap); re-run the ML-08
rule-vs-model race whenever a new forward month lands; **escalate to the warehouse model only if**
the rule's forward precision@50 falls below the baseline for two consecutive months. No heavy model
runs in production today - by design, because ML-08 showed one wasn't worth it yet.

In [4]:
# --- Compute the reference snapshot future runs will diff against ------------------
curve = vis.groupby("pos_band")["ctr"].median().round(3)
monitoring_baseline = {
    "expected_ctr_by_band": {int(k): float(v) for k, v in curve.items()},
    "actionable_fraction":  round(float((vis.archetype != "monitor").mean()), 3),
    "verify_tracking_count": int((vis.archetype == "verify_tracking").sum()),
    "clients_present":       int(vis.client_id.nunique()),
    "validated_precision_at_50": 0.84,          # ML-08 forward proxy, client-grouped
    "base_rate_reference":       0.53,          # ML-08 P(ctr_up)
    "curve_drift_trigger_pct": 25,
    "precision_floor_trigger": 0.53,
}
print("=== monitoring reference snapshot (stored in the exports) ===")
import json
print(json.dumps(monitoring_baseline, indent=2)[:900] + "\n...")
print(f"\nactionable fraction now: {monitoring_baseline['actionable_fraction']:.0%} | "
      f"verify_tracking now: {monitoring_baseline['verify_tracking_count']} | "
      f"clients now: {monitoring_baseline['clients_present']}")


=== monitoring reference snapshot (stored in the exports) ===
{
  "expected_ctr_by_band": {
    "1": 0.07,
    "2": 0.29,
    "3": 0.33,
    "4": 0.35,
    "5": 0.31,
    "6": 0.25,
    "7": 0.23,
    "8": 0.21,
    "9": 0.23,
    "10": 0.19,
    "11": 0.22,
    "12": 0.22,
    "13": 0.22,
    "14": 0.24,
    "15": 0.22,
    "16": 0.22,
    "17": 0.215,
    "18": 0.2,
    "19": 0.18,
    "20": 0.19
  },
  "actionable_fraction": 0.644,
  "verify_tracking_count": 8,
  "clients_present": 28,
  "validated_precision_at_50": 0.84,
  "base_rate_reference": 0.53,
  "curve_drift_trigger_pct": 25,
  "precision_floor_trigger": 0.53
}
...

actionable fraction now: 64% | verify_tracking now: 8 | clients now: 28


## 5. Exports for the paper

Three artifacts, and the git rules the card states:
- **`work/outputs/content_action_playbook.csv`** - the ranked queue. **Stays out of git** (leak-guard
  blocks `work/**/*.csv`); the notebook regenerates it every run. This is the file the paper's
  recommendations table is built from.
- **`work/figures/*.png`** - two reusable figures, **committed** (the paper embeds them).
- **`work/outputs/w07_playbook_metrics.json`** - the committed **receipt**: composition, limits, and
  the monitoring baseline the paper's numbers trace back to.

In [5]:
# --- Write the queue CSV, two figures, and the metrics receipt ---------------------
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# 1) Ranked queue CSV (regenerated, gitignored) -------------------------------------
OUT_COLS = ["rank", "content_id", "client_id", "archetype", "action", "reason_code",
            "missed_clicks_90d", "est_value_usd", "cpc", "avg_position", "pos_band",
            "impressions_90d", "clicks_90d", "ctr", "expected_ctr", "ctr_gap",
            "content_age_days", "days_since_last_update", "word_count",
            "thin_band_flag", "human_review_required", "auto_publish", "review_note"]
out = queue[OUT_COLS].copy()
for c in ["missed_clicks_90d", "ctr", "expected_ctr", "ctr_gap"]:
    out[c] = out[c].round(3)
CSV_PATH = "work/outputs/content_action_playbook.csv"
out.to_csv(CSV_PATH, index=False)
print(f"wrote {CSV_PATH}: {len(out):,} ranked actions (gitignored, regenerated each run)")

# 2a) Figure - archetype composition ------------------------------------------------
comp_plot = vis.archetype.value_counts().reindex(list(ACTIONS.keys())).fillna(0).astype(int)
COLORS = {"verify_tracking": "#c0392b", "stale_refresh": "#8e6fb0", "thin_expand": "#5b8def",
          "ctr_rewrite": "#2f9e6f", "striking_distance": "#e0a63c", "monitor": "#9aa3ad"}
fig, ax = plt.subplots(figsize=(7.2, 3.6))
ax.barh(comp_plot.index[::-1], comp_plot.values[::-1],
        color=[COLORS[a] for a in comp_plot.index[::-1]])
for i, (a, v) in enumerate(zip(comp_plot.index[::-1], comp_plot.values[::-1])):
    ax.text(v + max(comp_plot)*0.01, i, f"{v:,}", va="center", fontsize=9)
ax.set_title("Playbook queue composition by archetype (visible universe)", fontsize=11)
ax.set_xlabel("pages"); ax.margins(x=0.12)
for s in ["top", "right"]: ax.spines[s].set_visible(False)
fig.tight_layout(); fig.savefig("work/figures/w07_archetype_mix.png", dpi=120, bbox_inches="tight")
plt.close(fig)

# 2b) Figure - how far down to review (cumulative missed clicks vs rank) -------------
cum = np.cumsum(queue.missed_clicks_90d.values)
cum_share = cum / cum[-1]
fig, ax = plt.subplots(figsize=(7.2, 3.6))
ax.plot(np.arange(1, len(cum)+1), cum_share, color="#2f9e6f", lw=2)
for k in [50, 200, 500]:
    if k <= len(cum):
        ax.axvline(k, color="#9aa3ad", ls="--", lw=0.8)
        ax.text(k, 0.04, f" top {k}: {cum_share[k-1]:.0%}", rotation=90, va="bottom", fontsize=8, color="#555")
ax.set_title("Recoverable clicks are front-loaded: cumulative share vs queue depth", fontsize=11)
ax.set_xlabel("queue rank (pages reviewed, in order)"); ax.set_ylabel("share of total missed clicks")
ax.set_ylim(0, 1.02)
for s in ["top", "right"]: ax.spines[s].set_visible(False)
fig.tight_layout(); fig.savefig("work/figures/w07_review_depth_curve.png", dpi=120, bbox_inches="tight")
plt.close(fig)
print("wrote work/figures/w07_archetype_mix.png and work/figures/w07_review_depth_curve.png (committed)")

# 3) Metrics receipt (committed; safe aggregates only, NO ids) ----------------------
metrics = {
    "task": "ML-10 w07_action_playbook",
    "built_on": "ML-07 rule (won ML-08 at P@50; audited ML-09) - queue order is the validated part",
    "universe_pages": int(len(vis)),
    "universe_clients": int(vis.client_id.nunique()),
    "actionable_pages": int(len(queue)),
    "archetype_counts": {k: int((vis.archetype == k).sum()) for k in ACTIONS},
    "total_missed_clicks_90d": round(float(vis.missed_clicks_90d.sum()), 0),
    "est_value_usd_where_cpc_known": round(float(vis.est_value_usd.sum()), 2),
    "cpc_coverage": round(float((vis.cpc > 0).mean()), 3),
    "top50_pct_recoverable_clicks": round(float(cum_share[49]), 3),
    "claim_level": {"queue_order": "validated decision-support (P@50 0.84 vs base 0.53)",
                     "archetype_action": "directional guidance, not per-page validated"},
    "monitoring_baseline": monitoring_baseline,
    "exports": {"queue_csv": CSV_PATH + " (gitignored, regenerated)",
                "figures": ["work/figures/w07_archetype_mix.png", "work/figures/w07_review_depth_curve.png"]},
}
with open("work/outputs/w07_playbook_metrics.json", "w") as fh:
    json.dump(metrics, fh, indent=2)
print("wrote work/outputs/w07_playbook_metrics.json (committed receipt)")


wrote work/outputs/content_action_playbook.csv: 6,962 ranked actions (gitignored, regenerated each run)


wrote work/figures/w07_archetype_mix.png and work/figures/w07_review_depth_curve.png (committed)
wrote work/outputs/w07_playbook_metrics.json (committed receipt)


## Self-check

- [x] **Intended use** stated (§2): a 50-hour/week editor's prioritisation aid, decision-support, one portfolio/period.
- [x] **Ranked actions + reason codes** (§1): queue sorted by `missed_clicks_90d`, each row a reason code.
- [x] **Archetype -> action mapping** (§1): six archetypes, documented cascade, one action each.
- [x] **Decay/refresh insight** (§1): included and honestly sized - real but secondary here (old-but-fresh universe).
- [x] **Cost/value** (§1): `clicks x CPC` value proxy, secondary to the rank, CPC-coverage caveat stated.
- [x] **Human review + no-go list** (§3): per-archetype checks + 8 things that must never be automated.
- [x] **Monitoring / retrain triggers** (§4): five monitored signals + light re-fit rules + stored reference snapshot.
- [x] **Exports for the paper** (§5): queue CSV (gitignored), two committed figures, committed metrics JSON.
- [x] **Claims use safe language** - observed / measured / directional / decision-support; queue-order vs archetype-action kept separate.
- [ ] Runs top to bottom with no errors (Runtime -> Run all) - **confirm on your run**.
- [ ] Committed under `work/notebooks/`; figures under `work/figures/`; then submit the repo URL.

### The playbook in one honest paragraph (paper-ready)

On this portfolio, in this period, I **observed** that visible pages under-capturing clicks relative
to same-rank peers are the ones **worth reviewing first**, and a client-grouped forward check
**measured** that ranking at precision@50 ~0.84 against a ~0.53 base rate - a learned model did not
beat it, so the shipped tool is a transparent rule. The queue tags each page with a **directional**
archetype (rewrite / refresh / expand / promote) and a value estimate where CPC is known, but every
edit is **human-reviewed and never auto-published**, tracking-suspect pages are verified before any
content work, and the whole thing is **decision-support for one portfolio - not a claim about what
Google rewards**. Refresh, the paper's strongest portfolio lever, is real here but secondary,
because these pages are already old-but-freshly-maintained.